In [14]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

In [16]:
# Project paths
BASE_DIR = Path.cwd().parent

SPENDING_FILE = BASE_DIR / "data/raw/spending_profile.csv"
CARDS_FILE = BASE_DIR / "data/raw/credit_cards.csv"

REPORT_DIR = BASE_DIR / "reports"
FIGURE_DIR = BASE_DIR / "figures"

REPORT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

In [17]:
# Match spending categories to card reward-rate columns
CATEGORY_RATE_MAP = {
    "Groceries": "groceries_rate",
    "Dining": "dining_rate",
    "Gas": "gas_rate",
    "Travel": "travel_rate",
    "Bills": "bills_rate",
    "Other": "other_rate"
}


def main():

    # Load datasets
    spending = pd.read_csv(SPENDING_FILE)
    cards = pd.read_csv(CARDS_FILE)

    # Convert numeric columns
    spending["monthly_spending"] = pd.to_numeric(
        spending["monthly_spending"], errors="coerce"
    )

    numeric_columns = [
        "annual_fee",
        "welcome_bonus_value"
    ] + list(CATEGORY_RATE_MAP.values())

    for col in numeric_columns:
        cards[col] = pd.to_numeric(cards[col], errors="coerce")

    # Validate spending categories
    unsupported_categories = set(spending["category"].dropna()) - set(
        CATEGORY_RATE_MAP.keys()
    )

    if unsupported_categories:
        raise ValueError(
            f"Unsupported spending categories: {unsupported_categories}"
        )

    if spending["monthly_spending"].isna().any():
        raise ValueError("Missing or invalid spending amounts found.")

    if (spending["monthly_spending"] < 0).any():
        raise ValueError("Negative spending amounts found.")

    if cards[numeric_columns].isna().any().any():
        raise ValueError(
            "Missing card fees, reward rates, or bonus values found."
        )

    if (cards["annual_fee"] < 0).any().any():
        raise ValueError("Annual fees cannot be negative.")

    if (cards[list(CATEGORY_RATE_MAP.values())] < 0).any().any():
        raise ValueError("Reward rates cannot be negative.")

    if (cards[list(CATEGORY_RATE_MAP.values())] > 1).any().any():
        raise ValueError("Reward rates must be between 0 and 1.")

    if (cards["welcome_bonus_value"] < 0).any():
        raise ValueError("Welcome bonus values cannot be negative.")

    # Calculate annual spending by category
    spending["annual_spending"] = (
        spending["monthly_spending"] * 12
    )

    # Calculate rewards for every card and category
    detailed_results = []

    for _, card in cards.iterrows():

        for _, category in spending.iterrows():

            category_name = category["category"]
            annual_spending = category["annual_spending"]

            rate_column = CATEGORY_RATE_MAP[category_name]
            reward_rate = card[rate_column]

            annual_rewards = annual_spending * reward_rate

            detailed_results.append({
                "card_name": card["card_name"],
                "category": category_name,
                "annual_spending": annual_spending,
                "reward_rate": reward_rate,
                "annual_rewards": annual_rewards
            })

    rewards_detail = pd.DataFrame(detailed_results)

    # Summarize annual rewards for each card
    summary = (
        rewards_detail
        .groupby("card_name", as_index=False)
        .agg(
            total_annual_spending=("annual_spending", "sum"),
            total_annual_rewards=("annual_rewards", "sum")
        )
    )

    # Add card fees and welcome bonuses
    card_details = cards[
        ["card_name", "annual_fee", "welcome_bonus_value",
         "data_status"]
    ]

    summary = summary.merge(
        card_details,
        on="card_name",
        how="left"
    )

    # Calculate net values
    summary["ongoing_net_value"] = (
        summary["total_annual_rewards"] - summary["annual_fee"]
    )

    summary["first_year_net_value"] = (
        summary["ongoing_net_value"] +
        summary["welcome_bonus_value"]
    )

    # Rank cards by ongoing net value
    summary["ongoing_rank"] = (
        summary["ongoing_net_value"]
        .rank(method="min", ascending=False)
        .astype(int)
    )

    summary["first_year_rank"] = (
        summary["first_year_net_value"]
        .rank(method="min", ascending=False)
        .astype(int)
    )

    summary = summary.sort_values(
        "ongoing_net_value",
        ascending=False
    )

    # Save reports
    rewards_detail.to_csv(
        REPORT_DIR / "card_rewards_by_category.csv",
        index=False
    )

    summary.to_csv(
        REPORT_DIR / "card_rewards_summary.csv",
        index=False
    )

    # Display results
    print("\nCREDIT CARD REWARDS OPTIMIZER")
    print("ANNUAL REWARDS COMPARISON")
    print("-" * 60)

    display_columns = [
        "card_name",
        "total_annual_rewards",
        "annual_fee",
        "ongoing_net_value",
        "welcome_bonus_value",
        "first_year_net_value",
        "ongoing_rank"
    ]

    print(
        summary[display_columns].to_string(
            index=False,
            formatters={
                "total_annual_rewards": "${:,.2f}".format,
                "annual_fee": "${:,.2f}".format,
                "ongoing_net_value": "${:,.2f}".format,
                "welcome_bonus_value": "${:,.2f}".format,
                "first_year_net_value": "${:,.2f}".format
            }
        )
    )

    print("\nBest card for ongoing annual value:")

    best_card = summary.iloc[0]

    print(
        f"{best_card['card_name']}: "
        f"${best_card['ongoing_net_value']:,.2f}"
    )

    print("\nReports saved in the reports folder.")


if __name__ == "__main__":
    main()



CREDIT CARD REWARDS OPTIMIZER
ANNUAL REWARDS COMPARISON
------------------------------------------------------------
              card_name total_annual_rewards annual_fee ongoing_net_value welcome_bonus_value first_year_net_value  ongoing_rank
Wells Fargo Active Cash              $439.20      $0.00           $439.20             $100.00              $539.20             1
      Capital One Savor              $435.60      $0.00           $435.60             $200.00              $635.60             2
Chase Freedom Unlimited              $383.40      $0.00           $383.40             $200.00              $583.40             3

Best card for ongoing annual value:
Wells Fargo Active Cash: $439.20

Reports saved in the reports folder.
